# Часть 1. EDA и Feature Engineering

**Вариант 3:** классификация текстов классическими ML-моделями

**Датасет:** SMS Spam Collection (UCI)

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL  = "https://github.com/TiotioKun/spam-clf.git"
REPO_NAME = "spam-clf"

if "google.colab" in sys.modules:
    target = Path("/content") / REPO_NAME

    if target.exists():
        os.chdir(target / "notebooks")
        print("Репозиторий уже склонирован")
    else:
        r = subprocess.run(["git", "clone", REPO_URL, str(target)],
                           capture_output=True, text=True)
        if r.returncode == 0:
            os.chdir(target / "notebooks")
            print("Репозиторий склонирован")
        else:
            print("Клонировать не удалось, работаем в текущем каталоге.")
            print("Причина:", r.stderr.strip().splitlines()[-1] if r.stderr else "неизвестна")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightgbm", "wordcloud"], check=False)

print("Рабочий каталог:", Path.cwd())

FileNotFoundError: [Errno 2] No such file or directory: '/content/spam-clf/notebooks'

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, joblib, sklearn


sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11

# Корень проекта определяем автоматически:
# работает и в Colab (плоская структура /content),
# и в репозитории, где ноутбук лежит в notebooks/
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD

SRC_DIR   = ROOT / "src"
DATA_DIR  = ROOT / "data"
FIG_DIR   = ROOT / "reports" / "figures"
MODEL_DIR = ROOT / "models"
for d in (DATA_DIR, FIG_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# features.py ищем и в src/, и рядом с ноутбуком
sys.path.insert(0, str(SRC_DIR if SRC_DIR.exists() else ROOT))

RANDOM_STATE = 42

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

print("ROOT:", ROOT)
print("sklearn:", sklearn.__version__)

Почему этот датасет?

Легко обучается за секунды, плюс явный дисбаланс, поэтому есть что анализировать в EDA. Есть характерные лексические и синтаксические маркеры

In [ ]:
df = pd.read_csv(DATA_DIR / "spam.csv", encoding="latin-1").iloc[:, :2]
df.columns = ["label", "text"]
df["target"] = (df["label"] == "spam").astype(int)

print(f"{len(df)} сообщений, доля спама {df['target'].mean():.3f}")
df.head()

## 1.2 EDA

### Пропуски, дубликаты, базовая статистика

In [ ]:
print("Пропуски:\n", df.isna().sum(), "\n")
print("Полные дубликаты:", df.duplicated(subset=["text", "label"]).sum())

# Дубликаты убираем: одинаковые сообщения в train и test завышают метрики
before = len(df)
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
print(f"Было {before}, стало {len(df)}")

df["label"].value_counts()

### Распределение классов

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

counts = df["label"].value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette="Set2")
ax[0].set_title("Количество сообщений по классам")
ax[0].set_ylabel("Количество")
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 30, str(v), ha="center")

ax[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%",
          colors=sns.color_palette("Set2"), startangle=90)
ax[1].set_title("Доли классов")

save_fig("01_class_balance")
plt.show()

imbalance = counts.max() / counts.min()
print(f"Дисбаланс: {imbalance:.1f} : 1")
print("Вывод: accuracy как основная метрика непригодна — тривиальный")
print("классификатор 'всё ham' даст ~{:.1%}. Основная метрика — F1 по классу spam.".format(
    counts.max() / counts.sum()))

### Анализ длин текстов

In [ ]:
df["length"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(data=df, x="length", hue="label", bins=50, ax=ax[0],
             element="step", stat="density", common_norm=False)
ax[0].set_title("Распределение длин (символы)")

sns.boxplot(data=df, x="label", y="length", ax=ax[1], palette="Set2")
ax[1].set_title("Boxplot длин по классам")

sns.boxplot(data=df, x="label", y="n_words", ax=ax[2], palette="Set2")
ax[2].set_title("Boxplot количества слов")

save_fig("02_text_length")
plt.show()

df.groupby("label")[["length", "n_words"]].describe().T.round(1)

### Выбросы


In [ ]:
q1, q3 = df["length"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = df[df["length"] > upper]

print(f"Граница выброса по IQR: {upper:.0f} символов")
print(f"Выбросов: {len(outliers)} ({len(outliers)/len(df):.1%})")
print(outliers["label"].value_counts())
outliers[["label", "length", "text"]].head(3)

### Частотный анализ (WordCloud)

In [ ]:
!pip install wordcloud
from wordcloud import WordCloud
from features import STOP_WORDS

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i, lab in enumerate(["ham", "spam"]):
    text = " ".join(df.loc[df["label"] == lab, "text"].str.lower())
    wc = WordCloud(width=800, height=400, background_color="white",
                   stopwords=STOP_WORDS, colormap="viridis" if lab == "ham" else "inferno",
                   random_state=RANDOM_STATE).generate(text)
    ax[i].imshow(wc, interpolation="bilinear")
    ax[i].axis("off")
    ax[i].set_title(f"Частотные слова: {lab}", fontsize=14)

save_fig("03_wordclouds")
plt.show()

In [ ]:
# Топ-15 слов по классам — числовая версия того же анализа
from collections import Counter
import re

def top_words(series, n=15):
    words = re.findall(r"\b[a-z]{3,}\b", " ".join(series).lower())
    return Counter(w for w in words if w not in STOP_WORDS).most_common(n)

top = pd.DataFrame({
    "ham": [f"{w} ({c})" for w, c in top_words(df.loc[df.label == "ham", "text"])],
    "spam": [f"{w} ({c})" for w, c in top_words(df.loc[df.label == "spam", "text"])],
})
top

## 1.3 Feature Engineering

Все признаки считает модуль `src/features.py`. Здесь мы их только
проверяем и визуализируем — **логику признаков в ноутбуке не дублируем**,
иначе веб-сервис посчитает их иначе, чем обучение.

In [ ]:
from features import ManualFeatures, MANUAL_FEATURE_NAMES, build_feature_union

mf = ManualFeatures()
feats = mf.to_frame(df["text"])
feats["label"] = df["label"].values

print(f"Ручных признаков: {len(MANUAL_FEATURE_NAMES)}")
feats.head().T

In [ ]:
#как признаки разделяют классы
show = ["n_chars", "n_words", "avg_word_len", "lexical_diversity",
        "stopword_ratio", "n_punct", "upper_ratio", "digit_ratio", "n_exclamations"]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for axis, col in zip(axes.ravel(), show):
    sns.boxplot(data=feats, x="label", y=col, ax=axis, palette="Set2")
    axis.set_title(col)
    axis.set_xlabel("")

save_fig("04_manual_features")
plt.show()

In [ ]:
# Численная оценка разделяющей силы: точечно-бисериальная корреляция с таргетом


if "target" not in df.columns:
    positive = "spam"
    df["target"] = (df["label"] == positive).astype(int)
    print(f"Колонка target создана заново, доля класса '{positive}': {df['target'].mean():.3f}")

feats["target"] = df["target"].values

corr = (feats[MANUAL_FEATURE_NAMES]
        .corrwith(feats["target"])
        .sort_values(key=abs, ascending=False))

plt.figure(figsize=(7, 5))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette="coolwarm", legend=False)
plt.title("Корреляция ручных признаков с классом spam")
plt.xlabel("Корреляция")
save_fig("05_feature_correlation")
plt.show()

corr.round(3)

In [ ]:
# Матрица корреляций — ищем избыточные признаки
plt.figure(figsize=(10, 8))
sns.heatmap(feats[MANUAL_FEATURE_NAMES].corr(), cmap="coolwarm", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7}, square=True)
plt.title("Корреляции между ручными признаками")
save_fig("06_feature_corr_matrix")
plt.show()

## 1.4 Разделение выборки

Стратифицированное разделение 70/15/15 — доли классов сохраняются
во всех трёх выборках, что критично при дисбалансе 87/13.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df["text"].values, df["target"].values

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)

for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s}: {len(yy):5d}  доля spam = {yy.mean():.3f}")

np.savez(DATA_DIR / "split.npz",
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)
print("\nРазбиение сохранено в data/split.npz")

## 1.5 Сравнение наборов признаков

Обязательный пункт задания. Фиксируем модель (логистическая регрессия
с дефолтными параметрами) и меняем только набор признаков — так разница
в метриках объясняется признаками, а не моделью.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

rows = []

# Четыре набора признаков на одинаковой модели без взвешивания классов
for kind in ["manual", "count", "tfidf", "combined"]:
    pipe = Pipeline([
        ("features", build_feature_union(kind, max_features=3000)),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    rows.append({
        "Набор признаков": kind,
        "Кол-во признаков": pipe.named_steps["features"].transform(X_train[:1]).shape[1],
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
    })

# Пятая строка: тот же combined, но с учётом дисбаланса классов
pipe = Pipeline([
    ("features", build_feature_union("combined", max_features=3000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=RANDOM_STATE)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_val)
rows.append({
    "Набор признаков": "combined + balanced",
    "Кол-во признаков": pipe.named_steps["features"].transform(X_train[:1]).shape[1],
    "Accuracy": accuracy_score(y_val, pred),
    "Precision": precision_score(y_val, pred),
    "Recall": recall_score(y_val, pred),
    "F1": f1_score(y_val, pred),
})

res = pd.DataFrame(rows).set_index("Набор признаков")
res.round(4)

NameError: name 'build_feature_union' is not defined

In [ ]:
plt.figure(figsize=(9, 4.5))
res[["Accuracy", "Precision", "Recall", "F1"]].plot(kind="bar", ax=plt.gca(), rot=0)
plt.title("Качество на валидации при разных наборах признаков")
plt.ylabel("Значение метрики")
plt.ylim(0.7, 1.0)
plt.legend(loc="lower right")
save_fig("07_feature_sets_comparison")
plt.show()

best = res["F1"].idxmax()
print(f"Лучший набор по F1: {best} (F1 = {res.loc[best, 'F1']:.4f})")

# Часть 2. Обучение и тюнинг моделей

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier   # pip install lightgbm

from features import build_feature_union, MANUAL_FEATURE_NAMES

FEATURES = "combined"
MAX_FEATURES = 3000

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
    "LightGBM": LGBMClassifier(
        class_weight="balanced", random_state=RANDOM_STATE, verbose=-1),
}

def make_pipe(clf, kind=FEATURES, max_features=MAX_FEATURES):
    return Pipeline([
        ("features", build_feature_union(kind, max_features=max_features)),
        ("clf", clf),
    ])

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

def evaluate(pipe, X, y):
    pred = pipe.predict(X)
    proba = pipe.predict_proba(X)[:, 1]
    return {
        "Accuracy":  accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall":    recall_score(y, pred),
        "F1":        f1_score(y, pred),
        "ROC-AUC":   roc_auc_score(y, proba),
    }, pred

baseline, fitted, preds = [], {}, {}
for name, clf in models.items():
    t0 = time.time()
    pipe = make_pipe(clf)
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0

    metrics, pred = evaluate(pipe, X_val, y_val)
    baseline.append({"Модель": name, **metrics, "Время обучения, с": round(train_time, 2)})
    fitted[name], preds[name] = pipe, pred
    print(f"{name:20s} F1={metrics['F1']:.4f}  ({train_time:.1f} с)")

base_df = pd.DataFrame(baseline).set_index("Модель").sort_values("F1", ascending=False)
base_df.round(4)

### Матрицы ошибок

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for axis, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_val, pred)
    ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"]).plot(
        ax=axis, cmap="Blues", colorbar=False, values_format="d")
    f1 = f1_score(y_val, pred)
    axis.set_title(f"{name}\nF1 = {f1:.3f}", fontsize=11)
    axis.set_xlabel("Предсказание")
    axis.set_ylabel("Истина" if axis is axes[0] else "")

save_fig("08_confusion_baseline")
plt.show()

In [ ]:
plt.figure(figsize=(11, 4.5))
base_df[["Accuracy", "Precision", "Recall", "F1"]].plot(kind="bar", ax=plt.gca(), rot=15)
plt.title("Базовое качество моделей на валидации")
plt.ylabel("Значение метрики")
plt.ylim(0.6, 1.02)
plt.legend(loc="lower right", ncol=2)
save_fig("09_baseline_comparison")
plt.show()

print("Топ-3 по F1:", list(base_df.index[:3]))

## 2.2 Тюнинг гиперпараметров

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# В sklearn 1.8 параметр penalty объявлен устаревшим (удаление в 1.10).
# Проверяем версию, чтобы код работал в любом окружении.
SK = tuple(int(x) for x in sklearn.__version__.split(".")[:2])

if SK >= (1, 8):
    logreg_grid = {
        "clf__C": [0.01, 0.1, 0.5, 1, 5, 10, 50],
        "clf__l1_ratio": [0.0, 0.5, 1.0],   # 0 = L2, 1 = L1
        "clf__class_weight": ["balanced", None],
        "features__tfidf__max_features": [1000, 3000, 5000],
    }
else:
    logreg_grid = {
        "clf__C": [0.01, 0.1, 0.5, 1, 5, 10, 50],
        "clf__penalty": ["l1", "l2"],
        "clf__solver": ["liblinear"],
        "clf__class_weight": ["balanced", None],
        "features__tfidf__max_features": [1000, 3000, 5000],
    }

grids = {
    "LogisticRegression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), logreg_grid),
    "RandomForest": (
        RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE), {
            "clf__n_estimators": [100, 300, 500],
            "clf__max_depth": [None, 20, 50],
            "clf__min_samples_split": [2, 5, 10],
            "clf__class_weight": ["balanced", "balanced_subsample"],
            "features__tfidf__max_features": [1000, 3000],
        }),
    "LightGBM": (
        LGBMClassifier(random_state=RANDOM_STATE, verbose=-1), {
            "clf__n_estimators": [100, 300, 500],
            "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
            "clf__max_depth": [-1, 5, 10],
            "clf__subsample": [0.7, 0.85, 1.0],
            "clf__class_weight": ["balanced", None],
            "features__tfidf__max_features": [1000, 3000],
        }),
}

# Оставляем только те модели, что попали в топ-3 базового этапа
top3 = [m for m in base_df.index[:3] if m in grids]
print("Тюним:", top3)

In [ ]:
tuned, search_results = {}, {}

for name in top3:
    clf, grid = grids[name]
    t0 = time.time()
    search = RandomizedSearchCV(
        make_pipe(clf), grid, n_iter=20, scoring="f1", cv=5,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=0)
    search.fit(X_train, y_train)

    tuned[name] = search.best_estimator_
    search_results[name] = search
    print(f"\n{name}  ({time.time() - t0:.0f} с)")
    print(f"  CV F1: {search.best_score_:.4f}")
    for k, v in search.best_params_.items():
        print(f"  {k} = {v}")

### Сравнение до и после тюнинга

In [ ]:
rows = []
for name in top3:
    before, _ = evaluate(fitted[name], X_val, y_val)
    after,  _ = evaluate(tuned[name],  X_val, y_val)
    rows.append({"Модель": name, "Этап": "до тюнинга", **before})
    rows.append({"Модель": name, "Этап": "после тюнинга", **after})

cmp_df = pd.DataFrame(rows)
pivot = cmp_df.pivot(index="Модель", columns="Этап", values="F1")
pivot["Прирост"] = pivot["после тюнинга"] - pivot["до тюнинга"]
display(pivot.round(4))

plt.figure(figsize=(9, 4.5))
sns.barplot(data=cmp_df, x="Модель", y="F1", hue="Этап", palette="Set2")
plt.title("F1 на валидации до и после тюнинга")
plt.ylim(0.8, 1.0)
plt.xticks(rotation=10)
save_fig("10_tuning_before_after")
plt.show()

### Выбор финальной модели и честная оценка на тесте

In [ ]:
best_name = max(top3, key=lambda n: f1_score(y_val, tuned[n].predict(X_val)))
best_model = tuned[best_name]
print("Финальная модель:", best_name)

test_metrics, test_pred = evaluate(best_model, X_test, y_test)
val_metrics,  _         = evaluate(best_model, X_val,  y_val)

final = pd.DataFrame({"Валидация": val_metrics, "Тест": test_metrics}).T
display(final.round(4))

cm = confusion_matrix(y_test, test_pred)
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"{best_name} — тестовая выборка")
ax.set_xlabel("Предсказание"); ax.set_ylabel("Истина")
save_fig("11_confusion_final")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Ложных срабатываний (ham → spam): {fp}  — потерянные сообщения")
print(f"Пропущенного спама (spam → ham): {fn}  — попал во входящие")

## 2.3 Анализ важности признаков

In [ ]:
feat_names = [str(s) for s in best_model.named_steps["features"].get_feature_names_out()]
clf = best_model.named_steps["clf"]

if hasattr(clf, "coef_"):
    importance = pd.Series(clf.coef_[0], index=feat_names)
    kind_label = "Коэффициент (знак = в пользу класса)"
    top = importance.sort_values(key=abs, ascending=False).head(25).sort_values()
else:
    importance = pd.Series(clf.feature_importances_, index=feat_names)
    kind_label = "Важность признака"
    top = importance.nlargest(25).sort_values()

plt.figure(figsize=(8, 8))
colors = ["#d62728" if v > 0 else "#1f77b4" for v in top.values]
plt.barh(top.index, top.values, color=colors)
plt.title(f"Топ-25 признаков — {best_name}")
plt.xlabel(kind_label)
plt.axvline(0, color="black", lw=0.8)
save_fig("12_feature_importance")
plt.show()

In [ ]:
# Отдельно: вклад ручных признаков против TF-IDF
manual_mask = [n.startswith("manual__") for n in feat_names]
w = importance.abs()

summary = pd.DataFrame({
    "Группа": ["Ручные признаки", "TF-IDF"],
    "Количество": [sum(manual_mask), len(feat_names) - sum(manual_mask)],
    "Суммарный вес": [w[manual_mask].sum(), w[[not m for m in manual_mask]].sum()],
})
summary["Средний вес"] = summary["Суммарный вес"] / summary["Количество"]
summary["Доля веса, %"] = 100 * summary["Суммарный вес"] / summary["Суммарный вес"].sum()
display(summary.round(3))

print("Вывод: ручных признаков в ~{:.0f} раз меньше, но средний вес каждого".format(
    summary.loc[1, "Количество"] / summary.loc[0, "Количество"]))
print("выше в {:.1f} раза — они плотные и информативные,".format(
    summary.loc[0, "Средний вес"] / summary.loc[1, "Средний вес"]))
print("тогда как TF-IDF берёт количеством разреженных слабых сигналов.")

In [ ]:
# Топ-10 ручных признаков крупным планом
manual_imp = importance[[n for n in feat_names if n.startswith("manual__")]]
manual_imp.index = [n.replace("manual__", "") for n in manual_imp.index]

plt.figure(figsize=(7, 5))
mi = manual_imp.sort_values()
plt.barh(mi.index, mi.values,
         color=["#d62728" if v > 0 else "#1f77b4" for v in mi.values])
plt.title("Вклад ручных признаков в финальную модель")
plt.xlabel(kind_label)
plt.axvline(0, color="black", lw=0.8)
save_fig("13_manual_importance")
plt.show()

### Примеры верных и ошибочных предсказаний

In [ ]:
proba_test = best_model.predict_proba(X_test)[:, 1]
res = pd.DataFrame({
    "text": X_test, "true": y_test, "pred": test_pred, "proba": proba_test,
})
res["Ошибка"] = np.where(
    res.true == res.pred, "верно",
    np.where(res.pred == 1, "ложное срабатывание", "пропущенный спам"))

def show(mask, title, n=3):
    sub = res[mask].head(n)
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")
    for _, r in sub.iterrows():
        print(f"[p(spam) = {r.proba:.3f}]  {r.text[:150]}")

show((res.Ошибка == "верно") & (res.true == 1), "ВЕРНО распознанный спам")
show((res.Ошибка == "верно") & (res.true == 0), "ВЕРНО распознанный ham")
show(res.Ошибка == "ложное срабатывание", "ОШИБКА: ham принят за спам", 5)
show(res.Ошибка == "пропущенный спам", "ОШИБКА: спам принят за ham", 5)

In [ ]:
plt.figure(figsize=(9, 4.5))
for lab, name, color in [(0, "ham", "#2ca02c"), (1, "spam", "#d62728")]:
    sns.histplot(res.loc[res.true == lab, "proba"], bins=40, label=name,
                 color=color, alpha=0.6, stat="count")
plt.axvline(0.5, color="black", ls="--", lw=1, label="порог 0.5")
plt.title("Распределение предсказанной вероятности спама (тест)")
plt.xlabel("p(spam)"); plt.ylabel("Количество")
plt.legend()
save_fig("14_confidence_distribution")
plt.show()

grey = res[(res.proba > 0.3) & (res.proba < 0.7)]
print(f"В серой зоне 0.3–0.7: {len(grey)} сообщений ({len(grey)/len(res):.1%})")

## Сохранение модели

In [ ]:
path = MODEL_DIR / "best_model.joblib"
joblib.dump(best_model, path, compress=3)

meta = {
    "model_name": best_name,
    "feature_set": FEATURES,
    "n_features": len(feat_names),
    "best_params": {k: str(v) for k, v in search_results[best_name].best_params_.items()},
    "metrics_test": {k: round(float(v), 4) for k, v in test_metrics.items()},
    "manual_features": MANUAL_FEATURE_NAMES,
    "sklearn_version": sklearn.__version__,
}
import json
(MODEL_DIR / "model_meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Сохранено: {path} ({path.stat().st_size / 1024:.0f} КБ)")

# Проверка: модель загружается и работает на сыром тексте
loaded = joblib.load(path)
for t in ["WIN a FREE iPhone! Call 09012345678 NOW!!!",
          "hey, are we still on for lunch tomorrow?"]:
    p = loaded.predict_proba([t])[0, 1]
    print(f"  p(spam) = {p:.3f}  <- {t[:50]}")

In [ ]:
!zip -qr /content/results.zip /content/reports /content/models /content/features.py
from google.colab import files
files.download("/content/results.zip")